# 13 · Group-aware validation for histogram reliability models

**One controlled question:** does choosing the early-stopping iteration on held-out source-cell groups improve
reliability prediction on the existing outer cell test, compared with the current query validation policy?

Notebook 12 found a compact-information benefit on random query splits, but poor accepted-answer quality on held-out
cell groups. This notebook compares **validation policies**, not new feature sets, model families or gate algorithms.
The outer train/calibration/test split stays exactly as saved in notebook 12. Grouped validation necessarily also changes
the observations used for fitting; their counts and identities are reported. Neither policy refits on all outer train rows.

## Run on Databricks

1. Pull this branch in your personal Git folder. Run the installation and configuration cells first.
2. Normally **leave `run_dir` blank**. The latest completed **notebook 12** run containing the requested seeds and cell
   splits under your personal `output_root` is selected by modification time and printed. To reproduce a particular run,
   enter that complete result directory. Do not enter an 08d directory or a ZIP here.
3. Keep `seeds=0,1,2`. Base histogram rates, tree budget, thread count and other LightGBM settings are inherited from 12.
   There is no merge-rate or model-search widget in this experiment. For a small first check you may request seed `0` only.
4. Run All. Both policies are retrained for `merged_only`, `compact16`, and `full_source` (18 fits for three seeds).
   Existing models and results are preserved; each attempt creates a new `group-validation-...` directory.

The completed 12 directory and its original 08d input directory must remain accessible. `data_dir` and `histogram_dir`
are optional relocation overrides for the original 08d files; their hashes must still match. No personal path is hard-coded.

In [ ]:
%pip install lightgbm==4.6.0

In [ ]:
import ast
import base64
import hashlib
import importlib
import importlib.metadata
import json
import math
import os
import platform
import shutil
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4
from zipfile import ZipFile, ZIP_DEFLATED

import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy.ndimage import uniform_filter
from sklearn.metrics import roc_auc_score


def notebook_defaults():
    try:
        context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        root = context.notebookPath().get().split('/ml/notebooks/')[0]
        repo = root if root.startswith('/Workspace/') else '/Workspace' + root
        output = '/Workspace/Users/' + context.userName().get() + '/synopsislake-results'
    except (NameError, AttributeError):
        repo, output = str(Path.cwd()), str(Path(tempfile.gettempdir())/'synopsislake-results')
    return dict(repo_root=repo, output_root=output, run_dir='', data_dir='', histogram_dir='', seeds='0,1,2')


DEFAULTS = notebook_defaults()
try:
    for key, value in DEFAULTS.items():
        dbutils.widgets.text(key, value)
    get_config = dbutils.widgets.get
except NameError:
    def get_config(key):
        return os.environ.get(key.upper(), DEFAULTS[key])


VARIANTS = ('merged_only', 'compact16', 'full_source')
POLICIES = ('query', 'cell')
INNER_FRACTION = .10
EARLY_STOPPING_ROUNDS = 100


def load_settings():
    cfg = {key: str(get_config(key)).strip() for key in DEFAULTS}
    cfg['seeds'] = [int(x) for x in cfg['seeds'].split(',')]
    if not cfg['seeds'] or len(set(cfg['seeds'])) != len(cfg['seeds']):
        raise ValueError('Use distinct seeds from the completed notebook 12 run')
    return cfg

## Reuse existing implementation

Only pinned definition cells from the repository's notebook 12 are loaded into an isolated namespace. Its widgets,
training and plotting cells are never executed. Hashes use selected Python AST definitions so normal saved outputs,
formatting, notebook metadata and missing cell IDs do not cause a mismatch. Changed definitions fail explicitly.
When notebook files are not readable through the workspace filesystem, a read-only Jupyter export uses the signed-in
Databricks runtime identity. No personal Databricks token is embedded or requested.

Existing merged/source feature code, encoded float16 metadata, cohort rules, query parsing, error labels, estimate parity
and empirical calibration are reused unchanged. The preparation stage still constructs auxiliary 12 artifacts, but this
study trains only three variants and makes no runtime or storage claim.

In [ ]:
REUSE_SPECS = {'compact-04': None, 'compact-07': ['sha', 'save_json', 'choose_ids', 'resolve_source', 'import_existing_helpers'], 'compact-09': ['histogram_context', 'base_query', 'source_query', 'extract_query', 'frame_for_context'], 'compact-11': ['prepare_artifacts'], 'compact-13': ['evaluate_score', 'fixed_coverage']}
REUSE_AST_SHA256 = {'compact-04': '1ac8d26a90d6e402c1223960b898fee17f79e9cea33c7656a7b699c6b30da8d3', 'compact-07': '59ed8ee9fe9aa4d989d47bcdad2c42af5d8067ad777db1765b0a978884496d30', 'compact-09': '0b24e98832f4b742419f0a417f021533793f5dd850873d87ee4dae86b520df04', 'compact-11': '97c22976f16bf0223ec1f7053c1f0fa106a06f6f3d0f5620aee9dd401c0fb6de', 'compact-13': '98478f1eec1edc83eb37b3fa686fafa933c62ad85356842c86fec6c07440128f'}


def file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()


def save_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False, default=str))
    temporary.replace(path)


def selected_ast(document, cell_id, names):
    # Databricks exports can remove/rewrite cell IDs. Locate verified content instead of relying on IDs.
    matches = []
    for cell in document['cells']:
        if cell['cell_type'] != 'code':
            continue
        try:
            tree = ast.parse(''.join(cell['source']))
        except SyntaxError:
            continue  # Notebook magic is not a definition source.
        if names is not None:
            nodes = [node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name in names]
            if {node.name for node in nodes} != set(names):
                continue
            tree = ast.Module(body=nodes, type_ignores=[])
        else:
            allowed = (ast.Import, ast.ImportFrom, ast.Assign, ast.FunctionDef)
            if any(not isinstance(node, allowed) and not
                   (isinstance(node, ast.Expr) and isinstance(node.value, ast.Constant) and isinstance(node.value.value, str))
                   for node in tree.body):
                continue
        digest = hashlib.sha256(ast.dump(tree, include_attributes=False).encode()).hexdigest()
        if digest == REUSE_AST_SHA256[cell_id]:
            matches.append(tree)
    if len(matches) != 1:
        raise ValueError('Missing, changed or duplicate notebook 12 definitions: '+cell_id+'. Pull matching notebooks 12 and 13.')
    return matches[0]


def export_workspace_notebook(path):
    # Some Databricks runtimes expose Git notebooks only through the Workspace API.
    # Use the signed-in runtime credentials; no personal token is stored in the notebook.
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.workspace import ExportFormat
    client = WorkspaceClient()
    workspace_path = str(path)
    if workspace_path.startswith('/Workspace/'):
        workspace_path = workspace_path[len('/Workspace'):]
    candidates = [workspace_path]
    if workspace_path.endswith('.ipynb'):
        candidates.append(workspace_path[:-len('.ipynb')])
    errors = []
    for candidate in candidates:
        try:
            exported = client.workspace.export(path=candidate, format=ExportFormat.JUPYTER)
            if not exported.content:
                raise ValueError('Workspace export returned no content')
            return base64.b64decode(exported.content)
        except Exception as error:
            errors.append(f'{candidate}: {type(error).__name__}: {error}')
    raise ValueError('Cannot export notebook 12 with the runtime identity: '+'; '.join(errors))


def read_notebook_document(path):
    path = Path(path)
    try:
        raw = path.read_bytes()
        document = json.loads(raw)
        method = 'workspace filesystem'
    except (OSError, ValueError, UnicodeError):
        if not str(path).startswith('/Workspace/'):
            raise
        try:
            raw = export_workspace_notebook(path)
            document = json.loads(raw)
            method = 'Workspace API Jupyter export using runtime identity'
        except Exception as error:
            raise ValueError('Cannot read notebook 12 as a file or Workspace export. Check repo_root and notebook access. '+str(error)) from error
    return document, dict(path=str(path),read_method=method,document_sha256=hashlib.sha256(raw).hexdigest())


def load_reused_definitions(repo):
    path = Path(repo).resolve()/'ml/notebooks/12_compact_metadata_ablation.ipynb'
    document, provenance = read_notebook_document(path)
    namespace = {name: globals()[name] for name in
                 ('hashlib','importlib','json','math','os','platform','shutil','tempfile','time',
                  'datetime','timezone','Path','uuid4','ZipFile','ZIP_DEFLATED','lgb','np','pd','uniform_filter','roc_auc_score')}
    namespace['__name__'] = 'notebook12_verified_definitions'
    verified = {}
    for cell_id, names in REUSE_SPECS.items():
        tree = selected_ast(document, cell_id, names)
        digest = hashlib.sha256(ast.dump(tree, include_attributes=False).encode()).hexdigest()
        if digest != REUSE_AST_SHA256[cell_id]:
            raise ValueError('Notebook 12 definitions changed: '+cell_id+'. Pull matching notebooks 12 and 13.')
        exec(compile(tree, str(path)+':'+cell_id, 'exec'), namespace)
        verified[cell_id] = digest
    provenance['definition_ast_sha256'] = verified
    return namespace, provenance


def resolve_source12(cfg):
    if cfg['run_dir']:
        source = Path(cfg['run_dir']).expanduser().resolve()
    else:
        candidates = []
        for path in Path(cfg['output_root']).expanduser().glob('*/run.json'):
            info = json.loads(path.read_text())
            if (info.get('status') == 'complete' and info.get('study') == 'compact-quality'
                    and 'cell' in info.get('config', {}).get('splits', [])
                    and set(cfg['seeds']) <= set(info.get('config', {}).get('seeds', []))):
                candidates.append(path)
        if not candidates:
            raise FileNotFoundError('Set run_dir to a completed notebook 12 directory with all requested seeds and cell splits.')
        source = max(candidates, key=lambda path: path.stat().st_mtime_ns).parent
    prior = json.loads((source/'run.json').read_text())
    if prior.get('status') != 'complete' or prior.get('study') != 'compact-quality':
        raise ValueError('run_dir must be a completed notebook 12 result directory, not 08d, 09, 13 or a ZIP')
    if ('cell' not in prior['config']['splits'] or not set(cfg['seeds']) <= set(prior['config']['seeds'])):
        raise ValueError('The notebook 12 source does not contain all requested outer cell splits')
    required = ('splits.csv','quality_by_setting.csv','test_predictions.csv','models.csv','feature_definitions.json','input_manifest.json')
    for name in required:
        if not (source/name).is_file():
            raise FileNotFoundError('Missing notebook 12 artifact: '+str(source/name))
    print('Using notebook 12 run:', source, flush=True)
    return source, prior

## Freeze the outer test; vary training-only validation

| Policy | Inner validation selection from outer training queries |
|---|---|
| `query` | Every tenth sorted training query ID, exactly matching `gate_eval.fit` |
| `cell` | A deterministic SHA-256 order of training cell IDs, salted by seed; hold out ceil(10% of cells) |

All feature variants share the same inner assignment. The grouped policy requires zero shared cell IDs between fitting
and validation. The selected groups depend only on training IDs, not labels or calibration/test outcomes. Ten percent of
cells need not contain exactly ten percent of queries. Both fitting and validation memberships are exported.

**Validation selects the number of trees. Calibration selects the acceptance cutoff.** These are separate datasets and
decisions. Outer calibration and test IDs are copied from 12, never regenerated. The existing cohort excludes exact counts
below 10, including zero; relative error is `abs(estimate - truth) / truth`.

In [ ]:
def make_inner_assignments(outer_train, seed, policy):
    train = outer_train[['qId','cell']].drop_duplicates().sort_values('qId').reset_index(drop=True)
    if train.qId.duplicated().any() or train.cell.isna().any():
        raise ValueError('Each training query must have exactly one known source-cell ID')
    if policy == 'query':
        validation_ids = set(train.qId.to_numpy()[::10])
    elif policy == 'cell':
        cells = sorted(set(train.cell.astype(str)))
        if len(cells) < 2:
            raise ValueError('At least two training cell groups are required')
        ordered = sorted(cells, key=lambda cell: hashlib.sha256(f'13-inner-cell:{seed}:{cell}'.encode()).digest())
        n = min(len(cells)-1, max(1, math.ceil(INNER_FRACTION*len(cells))))
        held = set(ordered[:n])
        validation_ids = set(train.loc[train.cell.astype(str).isin(held), 'qId'])
    else:
        raise ValueError('Unknown validation policy: '+str(policy))
    train['role'] = np.where(train.qId.isin(validation_ids), 'validation', 'fit')
    if set(train.role) != {'fit','validation'}:
        raise ValueError('Both inner fit and validation must be nonempty')
    return train


def prepare_experiment(cfg, destination, state):
    source, prior = resolve_source12(cfg)
    namespace, reuse = load_reused_definitions(cfg['repo_root'])
    for name, digest in prior['source_code_sha256'].items():
        if file_sha(Path(cfg['repo_root'])/'ml'/name) != digest:
            raise ValueError('Code differs from notebook 12: '+name+'. Use the matching checkout.')
    versions = {name: importlib.metadata.version(name) for name in prior['package_versions']}
    if versions != prior['package_versions']:
        raise ValueError('Package versions differ from notebook 12; use the same Databricks runtime to isolate validation policy: '+str(versions))
    source08d = Path(prior['source_run'])
    if not (source08d/'run.json').is_file():
        raise FileNotFoundError('Notebook 12 source 08d directory is unavailable: '+str(source08d))
    if file_sha(source08d/'run.json') != prior['source_manifest_sha256']:
        raise ValueError('Source 08d manifest changed since notebook 12')
    inherited = dict(prior['config'])
    inherited.update(repo_root=cfg['repo_root'], output_root=cfg['output_root'], run_dir=str(source08d),
                     data_dir=cfg['data_dir'], histogram_dir=cfg['histogram_dir'], seeds=cfg['seeds'], splits=['cell'],
                     merge_rates=','.join(map(str, prior['config']['resolved_base_rates'])))
    preparation = destination/'preparation'; preparation.mkdir()
    tick = time.perf_counter()
    data = namespace['prepare_artifacts'](inherited, preparation, state)
    state['timings']['artifact_and_feature_preparation_seconds'] = time.perf_counter()-tick
    definitions = json.loads((source/'feature_definitions.json').read_text())
    for variant in VARIANTS:
        if data['definitions'][variant] != definitions[variant]:
            raise ValueError('Feature definition differs from notebook 12: '+variant)
    data['definitions'] = {variant: data['definitions'][variant] for variant in VARIANTS}
    data['frames'] = {variant: data['frames'][variant] for variant in VARIANTS}
    if inherited['resolved_base_rates'] != prior['config']['resolved_base_rates']:
        raise ValueError('Resolved physical base rates differ from notebook 12')
    input_identity = json.loads((source/'input_manifest.json').read_text())
    current_identity = json.loads((preparation/'input_manifest.json').read_text())
    if ({key: item['sha256'] for key,item in input_identity.items()} !=
            {key: item['sha256'] for key,item in current_identity.items()}):
        raise ValueError('Input identity differs from notebook 12')
    outer = pd.read_csv(source/'splits.csv')
    outer = outer[(outer.split_kind == 'cell') & outer.seed.isin(cfg['seeds'])].copy()
    if outer.duplicated(['seed','qId']).any():
        raise ValueError('Duplicate outer split identifiers')
    cohort = set(data['frames']['merged_only'].qId)
    for seed in cfg['seeds']:
        part = outer[outer.seed == seed]
        if set(part.qId) != cohort or set(part.split) != {'train','calibration','test'}:
            raise ValueError('Incomplete outer split in notebook 12')
        mapped = data['cells'].loc[part.qId].astype(str).to_numpy()
        if not np.array_equal(mapped, part.cell.astype(str).to_numpy()):
            raise ValueError('Source-cell assignment differs from notebook 12')
        groups = [set(part.loc[part.split == label,'cell']) for label in ('train','calibration','test')]
        if any(groups[i] & groups[j] for i in range(3) for j in range(i)):
            raise ValueError('Outer source-cell groups overlap')
    outer.to_csv(destination/'outer_splits.csv', index=False)
    assignments, inner_summary = [], []
    for seed in cfg['seeds']:
        training = outer[(outer.seed == seed) & (outer.split == 'train')]
        for policy in POLICIES:
            inner = make_inner_assignments(training, seed, policy)
            fit, validation = (inner[inner.role == label] for label in ('fit','validation'))
            overlap = set(fit.cell) & set(validation.cell)
            if policy == 'cell' and overlap:
                raise ValueError('Inner grouped validation overlaps fitting cells')
            assignments.append(inner.assign(seed=seed, validation_policy=policy))
            inner_summary.append(dict(seed=seed, validation_policy=policy, fit_queries=len(fit),
                validation_queries=len(validation), fit_cells=fit.cell.nunique(), validation_cells=validation.cell.nunique(),
                overlapping_cells=len(overlap), validation_query_fraction=len(validation)/len(inner),
                validation_sibling_share=float(validation.cell.isin(set(fit.cell)).mean())))
    inner = pd.concat(assignments, ignore_index=True)
    inner.to_csv(destination/'inner_validation_assignments.csv', index=False)
    pd.DataFrame(inner_summary).to_csv(destination/'inner_split_summary.csv', index=False)
    references = dict(quality=pd.read_csv(source/'quality_by_setting.csv'),
                      predictions=pd.read_csv(source/'test_predictions.csv'), models=pd.read_csv(source/'models.csv'))
    namespace['HELPERS'] = data['gate_helpers']
    state.update(source12_run=str(source), source12_manifest_sha256=file_sha(source/'run.json'),
        source12_artifact_sha256={name:file_sha(source/name) for name in
                                 ('splits.csv','quality_by_setting.csv','test_predictions.csv','models.csv','feature_definitions.json','input_manifest.json')},
        reuse_notebook12=reuse, inherited_training_config={key:inherited[key] for key in
              ('max_trees','num_threads','random_seed','resolved_base_rates')}, package_versions=versions,
        variants=list(VARIANTS), feature_counts={variant:len(data['definitions'][variant]) for variant in VARIANTS},
        inner_validation_fraction=INNER_FRACTION, early_stopping_rounds=EARLY_STOPPING_ROUNDS)
    save_json(destination/'feature_definitions.json', data['definitions'])
    return data, namespace, inherited, outer, inner, references

## Train, reproduce the baseline, and calibrate

The three inputs are unchanged: `merged_only` has 14 source-free features, `compact16` adds two metadata features,
and `full_source` adds seven source features (21 total). Every learned model excludes SA inputs.

Both policies use the inherited LightGBM parameters, seed and `early_stopping(100)`. The query policy calls the existing
fit helper directly. Its test scores, calibrated decisions, counts and cutoffs must reproduce notebook 12 before a policy
comparison is accepted. Model hashes are recorded separately; CPU/runtime versions can matter.

Each model's cutoff is selected using only the same outer calibration queries, with a 25% error limit and a 5% empirical
accepted-violation target. This is not a population guarantee. Test labels do not select trees or gates. Zero accepted
answers have undefined violation, not 0%. Fixed 5%/10% score-only ranking diagnostics retain entire ties and are separate
from deployment gates; a model with few trees may accept much more than the nominal diagnostic fraction.

In [ ]:
def train_and_evaluate(cfg, destination, state, data, namespace, inherited, outer, inner, references):
    quality, fixed, models, predictions, calibration_predictions, audits = [], [], [], [], [], []
    params = dict(data['gate_helpers'].PARAMS, n_estimators=inherited['max_trees'], n_jobs=inherited['num_threads'])
    state['model_parameters'] = params
    for seed in cfg['seeds']:
        outside = outer[outer.seed == seed]
        groups = {name:set(outside.loc[outside.split == name,'qId']) for name in ('train','calibration','test')}
        for policy in POLICIES:
            assignment = inner[(inner.seed == seed) & (inner.validation_policy == policy)]
            fit_ids = set(assignment.loc[assignment.role == 'fit','qId'])
            validation_ids = set(assignment.loc[assignment.role == 'validation','qId'])
            if fit_ids & validation_ids or fit_ids | validation_ids != groups['train']:
                raise ValueError('Inner assignment is not a partition of outer training queries')
            for variant in VARIANTS:
                names, frame = data['definitions'][variant], data['frames'][variant]
                training = frame[frame.qId.isin(groups['train'])]
                fit, validation = (training[training.qId.isin(ids)] for ids in (fit_ids,validation_ids))
                tick = time.perf_counter()
                if policy == 'query':
                    model = data['gate_helpers'].fit(training, names, seed, params=params, stop=EARLY_STOPPING_ROUNDS)
                else:
                    model = lgb.LGBMRegressor(random_state=seed, **params)
                    model.fit(fit[names], fit.label, eval_set=[(validation[names], validation.label)],
                              callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)])
                fit_seconds = time.perf_counter()-tick
                folder = destination/'models'/policy/f'seed_{seed}'/variant; folder.mkdir(parents=True)
                model.booster_.save_model(str(folder/'model.txt'))
                schema = dict(features=names, outer_split='cell', seed=seed, validation_policy=policy,
                    variant=variant, label='log(max(relative_error,0.001))', min_exact=10,
                    params=params, early_stopping_rounds=EARLY_STOPPING_ROUNDS, refit_on_outer_train=False)
                save_json(folder/'schema.json',schema)
                model_digest = file_sha(folder/'model.txt')
                losses = model.best_score_['valid_0']
                models.append(dict(seed=seed, validation_policy=policy, variant=variant, fit_seconds=fit_seconds,
                    best_iteration=int(model.best_iteration_), tree_count=model.booster_.num_trees(),
                    validation_loss_metric=next(iter(losses)), validation_loss=float(next(iter(losses.values()))),
                    fit_query_histogram_rows=len(fit),validation_query_histogram_rows=len(validation),
                    model_bytes=(folder/'model.txt').stat().st_size, model_sha256=model_digest))
                for base in data['base_names']:
                    part = frame[frame.histogram_key == base]
                    cal, test = (part[part.qId.isin(groups[label])].sort_values('qId') for label in ('calibration','test'))
                    tick = time.perf_counter()
                    cs = model.predict(cal[names], num_threads=inherited['num_threads'])
                    ts = model.predict(test[names], num_threads=inherited['num_threads'])
                    batch_seconds = time.perf_counter()-tick
                    if not np.isfinite(cs).all() or not np.isfinite(ts).all():
                        raise ValueError('Nonfinite model prediction')
                    evaluation = namespace['evaluate_score'](cal,test,cs,ts,epsilon=.25,delta=.05)
                    prefix = dict(split_kind='cell',seed=seed,validation_policy=policy,variant=variant,
                                  base_rate=data['rates'][base],physical_rate=data['rates'][base])
                    quality.append(dict(**prefix, **evaluation, score='learned',
                        score_space='predicted log-relative-error',calibration_count=len(cal),batch_prediction_seconds=batch_seconds))
                    fixed.extend(dict(**prefix,**item,score='learned') for item in namespace['fixed_coverage'](test,ts))
                    cutoff = -np.inf if evaluation['accept_none'] else evaluation['cutoff']
                    prediction = test[['qId','histogram_key','estimate','truth','relErr']].assign(
                        score_log=ts,accepted=ts<=cutoff,**prefix)
                    predictions.append(prediction)
                    calibration_predictions.append(cal[['qId','histogram_key','estimate','truth','relErr']].assign(
                        score_log=cs,accepted=cs<=cutoff,**prefix))
                    if policy == 'query':
                        old = references['predictions']
                        old = old[(old.split_kind == 'cell') & (old.seed == seed) & (old.variant == variant)
                                  & (old.base_rate == data['rates'][base])].sort_values('qId')
                        old_quality = references['quality']
                        old_quality = old_quality[(old_quality.split_kind == 'cell') & (old_quality.seed == seed)
                            & (old_quality.variant == variant) & (old_quality.base_rate == data['rates'][base])
                            & (old_quality.score == 'learned')]
                        old_model = references['models']
                        old_model = old_model[(old_model.split_kind == 'cell') & (old_model.seed == seed) & (old_model.variant == variant)]
                        if (len(old) != len(test) or len(old_quality) != 1 or len(old_model) != 1
                                or not np.array_equal(old.qId.to_numpy(),test.qId.to_numpy())):
                            raise ValueError('Incomplete notebook 12 baseline reference')
                        max_gap = float(np.max(np.abs(ts-old.score_log.to_numpy())))
                        score_match = bool(np.allclose(ts,old.score_log.to_numpy(),rtol=1e-9,atol=1e-10))
                        decision_match = bool(np.array_equal(ts<=cutoff,old.accepted.to_numpy()))
                        metric_match = (evaluation['accepted_count'] == old_quality.iloc[0].accepted_count and
                                        evaluation['violation_count'] == old_quality.iloc[0].violation_count)
                        cutoff_match = (evaluation['accept_none'] == bool(old_quality.iloc[0].accept_none) and
                            (evaluation['accept_none'] or np.isclose(evaluation['cutoff'],old_quality.iloc[0].cutoff,rtol=1e-10,atol=1e-12)))
                        audits.append(dict(seed=seed,variant=variant,base_rate=data['rates'][base],max_score_gap=max_gap,
                            scores_match=score_match,decisions_match=decision_match,counts_match=bool(metric_match),
                            cutoff_match=bool(cutoff_match),model_sha256_match=model_digest == old_model.iloc[0].model_sha256))
                        pd.DataFrame(audits).to_csv(destination/'baseline_reproduction.csv',index=False)
                        if not (score_match and decision_match and metric_match and cutoff_match):
                            raise ValueError('Query-validation baseline does not reproduce notebook 12; inspect baseline_reproduction.csv before interpreting a policy change')
                print('seed',seed,'validation',policy,variant,'trees',model.booster_.num_trees(),
                      'fit seconds',round(fit_seconds,2),flush=True)
    quality = pd.DataFrame(quality); fixed = pd.DataFrame(fixed)
    quality.to_csv(destination/'quality_by_setting.csv',index=False)
    fixed.to_csv(destination/'fixed_coverage.csv',index=False)
    pd.DataFrame(models).to_csv(destination/'models.csv',index=False)
    pd.concat(predictions,ignore_index=True).to_csv(destination/'test_predictions.csv',index=False)
    pd.concat(calibration_predictions,ignore_index=True).to_csv(destination/'calibration_predictions.csv',index=False)
    return quality, fixed

## Save a reproducible policy comparison

Compare each candidate with its matching seed, feature variant and base histogram. Read accepted good-answer counts,
violations, acceptance and AUC together. Aggregate cases across seeds/rates are descriptive and are not independent trials.

This experiment reuses an inspected test set. It can diagnose the validation-policy hypothesis, but it cannot establish
a final improvement claim. Source-cell IDs are inferred generating groups, not strict non-overlapping geographic regions.
Any settings selected after inspecting these results require fresh untouched grouped queries for final validation.

In [ ]:
def export_results(destination):
    with ZipFile(destination/'group_validation_results.zip','w',ZIP_DEFLATED) as archive:
        for path in destination.rglob('*'):
            if (path.is_file() and path.suffix in {'.csv','.json','.md','.png'}
                    and 'models' not in path.relative_to(destination).parts):
                archive.write(path,path.relative_to(destination))


def run_group_validation(cfg):
    destination = Path(cfg['output_root']).expanduser().resolve()/('group-validation-'+uuid4().hex[:12])
    destination.mkdir(parents=True,exist_ok=False)
    state = dict(status='running',study='group-validation',config=cfg,timings={},
        started_utc=datetime.now(timezone.utc).isoformat(),python=platform.python_version(),epsilon=.25,delta=.05,
        notes=['This is an exploratory validation-policy comparison, not a new generalization claim.',
               'Outer cell train/calibration/test IDs and input/features/LightGBM settings are inherited from notebook 12.',
               'Both policies retrain; original query validation is checked against saved notebook 12 predictions and gates.',
               'Cell validation also changes the fitting observations; realized query/cell counts are reported.',
               'No refit on the full outer training set is performed after early stopping, matching the existing workflow.',
               'Group assignment uses training IDs only, never calibration/test labels or outcomes.',
               'Calibration independently chooses each model gate with the same 25% error limit and 5% empirical risk target.',
               'The reused preparation also builds auxiliary notebook 12 representations; only three variants are trained here.',
               'No runtime microbenchmark, E2E speedup, exact fallback or storage improvement is evaluated here.',
               'Inferred source-cell groups are not strict geographic separation or a new dataset.',
               'Seeds and merge rates can repeat queries; descriptive cases are not independent trials.'])
    save_json(destination/'run.json',state); started = time.perf_counter()
    try:
        data,namespace,inherited,outer,inner,references = prepare_experiment(cfg,destination,state)
        save_json(destination/'run.json',state)
        tick = time.perf_counter()
        quality,fixed = train_and_evaluate(cfg,destination,state,data,namespace,inherited,outer,inner,references)
        state['timings']['training_calibration_evaluation_seconds'] = time.perf_counter()-tick
        keys = ['seed','variant','base_rate']
        joined = quality[quality.validation_policy == 'cell'].merge(
            quality[quality.validation_policy == 'query'],on=keys,suffixes=('_cell','_query'),validate='one_to_one')
        pairs = joined[keys].copy()
        for metric in ('acceptance','violation','good_accepted_count','good_recall','auc'):
            pairs[metric+'_query'] = joined[metric+'_query']
            pairs[metric+'_cell'] = joined[metric+'_cell']
            pairs[metric+'_delta'] = joined[metric+'_cell']-joined[metric+'_query']
        pairs.to_csv(destination/'paired_deltas.csv',index=False)
        summary = quality.groupby(['validation_policy','variant'],as_index=False).agg(
            settings=('seed','size'),acceptance_mean=('acceptance','mean'),acceptance_min=('acceptance','min'),
            acceptance_max=('acceptance','max'),violation_mean_defined=('violation','mean'),
            zero_acceptance_settings=('accepted_count',lambda x:int((x==0).sum())),
            settings_above_target=('violation',lambda x:int((x>.05).sum())),
            accepted_cases=('accepted_count','sum'),violation_cases=('violation_count','sum'),
            good_accepted_cases=('good_accepted_count','sum'),auc_mean=('auc','mean'))
        summary['pooled_case_violation'] = np.where(summary.accepted_cases>0,
            summary.violation_cases/summary.accepted_cases,np.nan)
        summary.to_csv(destination/'summary.csv',index=False)
        fixed.groupby(['validation_policy','variant','nominal_coverage'],as_index=False).agg(
            settings=('seed','size'),violation_mean=('violation','mean'),actual_coverage_mean=('actual_coverage','mean'),
            actual_coverage_min=('actual_coverage','min'),actual_coverage_max=('actual_coverage','max'),
            tie_extra_cases=('tie_extra_count','sum')).to_csv(destination/'fixed_coverage_summary.csv',index=False)
        (destination/'analysis_en.md').write_text(
            '# Group-aware validation experiment\n\n'
            'Compare query and cell validation within the SAME outer cell split, seed, base rate and feature variant. '
            'Use quality_by_setting.csv and paired_deltas.csv; acceptance gains must be read with accepted violations '
            'and good-answer counts. Zero acceptance has undefined risk. Descriptive means can hide individual failures.\n\n'
            'baseline_reproduction.csv checks the retrained query-validation policy against notebook 12. '
            'inner_validation_assignments.csv and inner_split_summary.csv expose fitting/validation IDs, group overlap '
            'and realized fractions. Cell validation changes which observations fit the model; this is a policy comparison, '
            'not a controlled comparison with identical fitting observations. Neither policy refits on all outer train rows.\n\n'
            'Validation selects the early-stopping iteration. The separate outer calibration set selects the gate. '
            'Fixed 5%/10% coverage diagnostics select using test scores only and retain complete ties; actual coverage '
            'can be much larger for a low-tree model. These diagnostic thresholds are not deployment gates.\n\n'
            'The 08d cohort excludes exact counts below 10, including zero. All learned variants exclude SA inputs. '
            'No new features or model families are introduced. Reused notebook 12 preparation costs are setup costs, '
            'not query latency. This experiment makes no runtime, storage or E2E claim.\n\n'
            'The test has already been inspected. Any apparent improvement remains exploratory and requires fresh '
            'untouched grouped queries before a final claim. Source-cell groups are inferred from query size and centre; '
            'they are not strict non-overlapping geographic regions. Repeated seed/rate cases are not independent trials.\n')
        state.update(status='complete',wall_seconds=time.perf_counter()-started,
                     finished_utc=datetime.now(timezone.utc).isoformat())
        save_json(destination/'run.json',state); export_results(destination)
        print('COMPLETE:',destination,flush=True)
        return destination,quality,summary
    except BaseException as error:
        state.update(status='failed',error=f'{type(error).__name__}: {error}',wall_seconds=time.perf_counter()-started)
        save_json(destination/'run.json',state)
        print('FAILED; diagnostics preserved:',destination,flush=True)
        raise

## Execute

If preparation or baseline reproduction fails, the failed run's `run.json` and available audit CSVs are preserved.
Fix the mismatch before interpreting the comparison; there is no automatic alternative model or threshold.

In [ ]:
# RUN_GROUP_VALIDATION
config = load_settings()
result_dir,quality_by_setting,summary = run_group_validation(config)
display(summary) if 'display' in globals() else print(summary.to_string(index=False))

## Inspect and share

- `quality_by_setting.csv`, `paired_deltas.csv`: frozen-gate quality and cell-minus-query policy differences.
- `fixed_coverage.csv`: score-ranking diagnostics, including actual coverage and extra tied cases.
- `inner_validation_assignments.csv`, `inner_split_summary.csv`, `outer_splits.csv`: exact memberships and group overlap.
- `baseline_reproduction.csv`: reproduction of notebook 12's query-validation policy on outer cell tests.
- `models.csv`: tree count, validation loss, fitting time, model bytes and hashes.
- `calibration_predictions.csv`, `test_predictions.csv`, `feature_definitions.json`, `run.json`: reproducibility records.

Send **`group_validation_results.zip`** after completion. It contains CSVs, English notes, provenance and figures, while
model weights and histogram payloads stay in the result directory. The two panels distinguish frozen calibration gates
from a nominal 10% ranking diagnostic; complete ties can change the latter's actual selection fraction.

In [ ]:
# PLOT_GROUP_VALIDATION
import matplotlib.pyplot as plt
fixed = pd.read_csv(Path(result_dir)/'fixed_coverage.csv')
fig,axes = plt.subplots(1,2,figsize=(12,4))
colors = dict(merged_only='tab:blue',compact16='tab:orange',full_source='tab:green')
for (policy,variant),part in quality_by_setting.groupby(['validation_policy','variant']):
    marker = 'o' if policy=='query' else '^'
    label = f'{variant} / {policy} validation'
    axes[0].scatter(part.acceptance*100,part.violation*100,label=label,marker=marker,color=colors[variant],alpha=.8)
    subset = fixed[(fixed.validation_policy==policy)&(fixed.variant==variant)&(fixed.nominal_coverage==.1)]
    axes[1].scatter(subset.actual_coverage*100,subset.violation*100,label=label,marker=marker,color=colors[variant],alpha=.8)
for axis in axes:
    axis.axhline(5,color='black',linestyle='--',linewidth=1)
    axis.grid(alpha=.2)
    axis.set(xlabel='Accepted queries (%)',ylabel='Violation among accepted (%)')
axes[0].set_title('Frozen calibration gates on outer cell test')
axes[1].set_title('Nominal 10% ranking diagnostic: complete ties')
axes[0].legend(fontsize=7,bbox_to_anchor=(0,-.2),loc='upper left',ncol=2)
fig.tight_layout()
fig.savefig(Path(result_dir)/'validation_policy_quality.png',dpi=160,bbox_inches='tight')
plt.show()
export_results(Path(result_dir))
print('Upload:',Path(result_dir)/'group_validation_results.zip')